In [1]:
import polars as pl
import polars.selectors as cs
import duckdb
import orbital
import sqlglot
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from clean_sql import clean_sql

In [2]:
# get data
with duckdb.connect('../dev.duckdb') as con:
    df_feat = con.table("raw_feat").pl()
    df_targ = con.table("raw_targ").pl()

# prep for modeling (order, split)
df = df_targ.join(df_feat, how = "left", on = "customer_id")
X = df.drop("customer_id", "churn")
y = df.select("churn").to_numpy().ravel()

In [3]:
# build pipeline(s)

## feature pipeline does OneHotEncoding on all string columns (all are low/known cardinality)
## orbital can create some very verbose variable names (for uniqueness) so we clean those up some
cols_str = X.select( cs.string() ).columns
onho_enc = ('oh', OneHotEncoder(sparse_output = False), cols_str)
ppl_feat = Pipeline([
  ("encoder", ColumnTransformer([onho_enc], remainder='passthrough'))
]).set_output(transform="polars")
X_tran = ppl_feat.fit_transform(X, y)
X_tran.columns = [c.replace(' ','_').replace('-','_').replace('(','').replace(')','') for c in X_tran.columns]

## training pipeline fits actual random forest model
ppl_rafo = Pipeline([
  ("prep", ColumnTransformer([], remainder='passthrough')),
  ("rf", RandomForestClassifier(max_depth = 3, n_estimators = 10, random_state=123))
])
ppl_rafo.fit(X_tran, y)

## save out predictions for comparison
df_preds_py = pl.DataFrame({
  'customer_id': df.select('customer_id').to_numpy().ravel(),
  'pred': ppl_rafo.predict_proba(X_tran)[:,1]
})
df_preds_py.write_csv('preds_py.csv')

This is the part where you actually check if your model is any good. That is not my problem at the moment. =)

In [4]:
# convert to orbital

tbl = "TBL_REF" # placeholder replaced in cleaning

## creating mapping of source data types to orbital types 
type_map = {
    pl.String:orbital.types.StringColumnType(),
    pl.Int32:orbital.types.Int32ColumnType(),
    pl.Float64:orbital.types.DoubleColumnType()
}
dict_feat = {e: type_map.get(t) for e, t in zip(X.columns, X.dtypes)}
dict_rafo = {e: type_map.get(t) for e, t in zip(X_tran.columns, X_tran.dtypes)}

## features
orb_ppl_feat = orbital.parse_pipeline(ppl_feat, features=dict_feat)
sql_raw_feat = orbital.export_sql(tbl, orb_ppl_feat, dialect="duckdb")

## scoring
orb_ppl_rafo = orbital.parse_pipeline(ppl_rafo, features=dict_rafo)
sql_raw_pred = orbital.export_sql(tbl, orb_ppl_rafo, dialect="duckdb")

c:\Users\emily\Desktop\orbital-exploration\.venv\Lib\site-packages\orbital\translation\steps\trees\classifier.py:135: FutureWarning: `case` is deprecated as of v10.0.0, removed in v11.0; use ibis.cases()
  ibis.case().when(condition, t_val).else_(f_val).end()
c:\Users\emily\Desktop\orbital-exploration\.venv\Lib\site-packages\orbital\translation\steps\trees\classifier.py:157: FutureWarning: `case` is deprecated as of v10.0.0, removed in v11.0; use ibis.cases()
  ibis.case()


In [5]:
# clean up resulting SQL

## map long orbital names into simpler names sklearn produces
dict_renm = dict(zip(
    [e.alias for e in sqlglot.parse_one(sql_raw_feat)],
    X_tran.columns
))

## use custom converters
sql_fmt_feat = clean_sql(sql_raw_feat, 'raw_feat', col_id = 'customer_id', cols_renm = dict_renm)
sql_fmt_pred = clean_sql(sql_raw_pred, 'prep_feat', col_id = 'customer_id')

In [6]:
# write to dbt model

with open("../models/churn_model/prep_feat.sql", "w") as file:

    config = '{{ config( materialized="view") }}'
    file.writelines([config, '\n\n', sql_fmt_feat])

with open("../models/churn_model/pred_churn.sql", "w") as file:
    
    config = '{{ config( materialized="view") }}'
    file.writelines([config, '\n\n', sql_fmt_pred])